# `sr_gauss_all_test` — Interactive PySR Run

A scaled-down version of the `sr_gauss_all` production run for interactive exploration.
Uses the same inputs, operators, and loss function, but with fewer samples, populations,
and a short timeout so it returns equations in seconds rather than hours.

| Parameter | Production | Test |
|-----------|-----------|------|
| `subsetsize` | 10,000 | 500 |
| `populations` | 50 | 10 |
| `populationsize` | 150 | 30 |
| `targettotal` | 500,000 | 2,000 |
| `procs` | 50 | 4 |
| `timeout` | 5.5 hr | 120 s |

In [ ]:
import os
import sys
import copy
import tempfile
import shutil
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd

from scripts.utils import Config
from scripts.models.sr.train import load_data, subsample, fit, save, select_pareto_elbow

## Configuration

Build a scaled-down copy of the `sr_gauss_all` run config.

In [ ]:
config  = Config()
sr      = copy.deepcopy(config.sr)

RUNNAME    = 'sr_gauss_all_test'
RUNCONFIG  = sr['runs']['sr_gauss_all']          # same inputs as production
FIELDVARS  = RUNCONFIG['fieldvars']              # ['rh', 'thetae', 'thetaestar']
LOCALVARS  = RUNCONFIG['localvars']              # ['lf', 'lhf', 'shf']
PREDICTORS = FIELDVARS + LOCALVARS

# --- scaled-down search parameters ---
SUBSETSIZE = 500
PROCS      = 4
TIMEOUT    = 120       # seconds

sr['searchparams']['populations']       = 10
sr['searchparams']['populationsize']    = 30
sr['searchparams']['targettotal']       = 2000
sr['searchparams'].pop('iterations', None)   # targettotal takes precedence
sr['searchparams']['batchsize']         = 50

sp       = sr['searchparams']
popcount = sp['populations']
iterseff = sp['targettotal'] // popcount

print(f'Predictors : {PREDICTORS}')
print(f'Populations: {popcount}  |  Iters/population: {iterseff}  |  Workers: {PROCS}  |  Timeout: {TIMEOUT}s')

## Load Data

Load normalized train and validation splits, kernel-integrate the 3-D field variables
using the learned `param_gauss_all` weights (averaged across seeds), then concatenate
the scalar local variables. This exactly mirrors the production `load_data` call.

In [ ]:
print('Loading train split...')
xtrain, ytrain, refdatrain, vmtrain = load_data('train', RUNCONFIG, config, time_offset=0)

print('Loading valid split...')
xvalid, yvalid, _, vmvalid = load_data('valid', RUNCONFIG, config,
                                        time_offset=int(refdatrain.sizes['time']))

xfit = pd.concat([xtrain[vmtrain], xvalid[vmvalid]]).reset_index(drop=True)
yfit = np.concatenate([ytrain[vmtrain], yvalid[vmvalid]])
del xtrain, xvalid, ytrain, yvalid, refdatrain

print(f'Combined finite samples: {len(xfit):,}')
xfit.describe()

## Subsample

Draw ~500 samples with proportional coverage of the precipitation distribution
(one-decade log10 bins from 0.0001–100 mm, plus a dry bin).

In [ ]:
xsub, ysub = subsample(xfit, yfit, subsetsize=SUBSETSIZE, seed=sr['seed'])
del xfit, yfit

print(f'Subset size: {len(xsub):,} samples  ×  {xsub.shape[1]} predictors')
xsub.describe()

## Fit PySR

Runs the symbolic regression search. With the test parameters this typically
finishes in under 2 minutes. Julia startup adds ~1–2 minutes on first run.

In [ ]:
tmpdir = tempfile.mkdtemp(prefix='pysr_test_')
try:
    model = fit(xsub, ysub, PREDICTORS, sr, PROCS, TIMEOUT, tmpdir)
finally:
    shutil.rmtree(tmpdir, ignore_errors=True)

print('Search complete.')

## Pareto Frontier

All discovered equations sorted by complexity, with the elbow selection highlighted.

In [ ]:
equations = model.equations_.sort_values('complexity').reset_index(drop=True)
best      = select_pareto_elbow(equations)

display_cols = ['complexity', 'loss', 'equation']
styled = equations[display_cols].style.apply(
    lambda row: ['background-color: #d4edda' if row.name == best.name else '' for _ in row],
    axis=1
)
display(styled)

In [ ]:
print('=== Elbow-selected equation ===')
print(f'  {best["equation"]}')
print(f'  Complexity : {best["complexity"]}')
print(f'  Train loss : {best["loss"]:.6f}')

## Save (optional)

Uncomment to persist the test model to `models/sr/` alongside the production runs.
The output files will be named `sr_gauss_all_test_pareto.pkl` and `sr_gauss_all_test_equations.csv`.

In [ ]:
# save(model, RUNNAME, config)